# ✅ Solution 4: Analisis Distribusi Gaji Perusahaan

**Approach**: Per-departemen analysis + overall company stats + business recommendation  
**Key Insight**: Gaji perusahaan SELALU right-skewed; melaporkan mean gaji = menipu mayoritas karyawan.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import trim_mean

np.random.seed(42)

gaji_engineering = np.concatenate([
    np.random.normal(8, 1.5, 30), np.random.normal(15, 2, 20),
    np.random.normal(25, 3, 8), np.array([45, 55, 80])
]).clip(5, 100)

gaji_sales = np.concatenate([
    np.random.normal(6, 1, 35), np.random.normal(12, 2, 10),
    np.array([35, 45, 60, 70, 90])
]).clip(4, 100)

gaji_operations = np.random.normal(9, 1.5, 40).clip(6, 15)
gaji_executive = np.array([80, 95, 120, 150, 200])

df_gaji = pd.DataFrame({
    'gaji': np.concatenate([gaji_engineering, gaji_sales, gaji_operations, gaji_executive]),
    'departemen': (
        ['Engineering'] * len(gaji_engineering) + ['Sales'] * len(gaji_sales) +
        ['Operations'] * len(gaji_operations) + ['Executive'] * len(gaji_executive)
    )
})

In [ ]:
# SOLUSI TODO 1: Statistik per departemen
def dept_stats(group):
    return pd.Series({
        'n_karyawan': len(group),
        'mean': group.mean(),
        'median': group.median(),
        'trimmed_10pct': trim_mean(group.values, 0.1),
        'skewness': group.skew(),
        'min': group.min(),
        'max': group.max(),
    })

df_summary = df_gaji.groupby('departemen')['gaji'].apply(dept_stats).unstack()
df_summary['gap_persen'] = ((df_summary['mean'] - df_summary['median']) / df_summary['median'] * 100)
df_summary['rekomendasi'] = df_summary['skewness'].apply(
    lambda s: 'Median' if s > 1.0 else ('Trimmed Mean' if s > 0.5 else 'Mean')
)

print('Statistik Gaji per Departemen (juta Rp/bulan):')
display(df_summary.round(2))

In [ ]:
# SOLUSI TODO 2: Analisis keseluruhan
gaji_all = df_gaji['gaji']

mean_all = gaji_all.mean()
median_all = gaji_all.median()
trim_all = trim_mean(gaji_all.values, 0.1)
pct_below_mean = (gaji_all < mean_all).mean() * 100

print('=== Statistik Keseluruhan Perusahaan ===')
print(f'  Total karyawan : {len(gaji_all)}')
print(f'  Mean gaji      : Rp {mean_all:.2f} juta/bulan')
print(f'  Median gaji    : Rp {median_all:.2f} juta/bulan')
print(f'  Trimmed 10%    : Rp {trim_all:.2f} juta/bulan')
print(f'  Skewness       : {gaji_all.skew():.4f}')
print()
print(f'⚠️ {pct_below_mean:.1f}% karyawan gajinya DI BAWAH "rata-rata" (mean)!')
print(f'   Ini berarti mean BUKAN representasi gaji mayoritas karyawan.')

In [ ]:
# SOLUSI TODO 3: Visualisasi lengkap
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# [0,0] Boxplot per departemen
dept_order = ['Operations', 'Sales', 'Engineering', 'Executive']
sns.boxplot(data=df_gaji, x='departemen', y='gaji', order=dept_order, ax=axes[0,0], palette='Set2')
axes[0,0].set_title('Boxplot Gaji per Departemen', fontweight='bold')
axes[0,0].set_ylabel('Gaji (juta Rp)')
axes[0,0].set_xlabel('')

# [0,1] Histogram keseluruhan
axes[0,1].hist(gaji_all, bins=30, color='#607D8B', edgecolor='black', alpha=0.7)
axes[0,1].axvline(mean_all, color='red', linestyle='--', lw=2, label=f'Mean = {mean_all:.1f}')
axes[0,1].axvline(median_all, color='green', linestyle='-', lw=2, label=f'Median = {median_all:.1f}')
axes[0,1].axvline(trim_all, color='orange', linestyle=':', lw=2, label=f'Trim 10% = {trim_all:.1f}')
axes[0,1].set_title('Distribusi Gaji Keseluruhan', fontweight='bold')
axes[0,1].set_xlabel('Gaji (juta Rp)')
axes[0,1].legend()

# [1,0] Bar chart mean vs median per departemen
x = range(len(dept_order))
width = 0.3
axes[1,0].bar([i-width for i in x], [df_summary.loc[d, 'mean'] for d in dept_order], 
             width, label='Mean', color='red', alpha=0.7)
axes[1,0].bar(x, [df_summary.loc[d, 'median'] for d in dept_order],
             width, label='Median', color='green', alpha=0.7)
axes[1,0].bar([i+width for i in x], [df_summary.loc[d, 'trimmed_10pct'] for d in dept_order],
             width, label='Trim 10%', color='orange', alpha=0.7)
axes[1,0].set_xticks(x)
axes[1,0].set_xticklabels(dept_order, fontsize=9)
axes[1,0].set_title('Mean vs Median vs Trimmed Mean', fontweight='bold')
axes[1,0].set_ylabel('Gaji (juta Rp)')
axes[1,0].legend()

# [1,1] Violin plot
sns.violinplot(data=df_gaji[df_gaji['departemen'] != 'Executive'], 
               x='departemen', y='gaji', ax=axes[1,1], palette='Set2',
               order=['Operations', 'Sales', 'Engineering'])
axes[1,1].set_title('Violin Plot (tanpa Executive)\nMenunjukkan bentuk distribusi', fontweight='bold')
axes[1,1].set_ylabel('Gaji (juta Rp)')
axes[1,1].set_xlabel('')

plt.suptitle('Analisis Distribusi Gaji — TechCorp', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# SOLUSI TODO 4: Rekomendasi untuk HR
print('=' * 70)
print('📋 REKOMENDASI UNTUK HR DIRECTOR')
print('=' * 70)
print()
print(f'1. GAJI "RATA-RATA" PERUSAHAAN:')
print(f'   ❌ Jangan pakai mean (Rp {mean_all:.1f} juta) — menipu!')
print(f'      {pct_below_mean:.0f}% karyawan gajinya di bawah angka ini.')
print(f'   ✅ Pakai median (Rp {median_all:.1f} juta) — mewakili karyawan tipikal.')
print()
print(f'2. RISIKO JIKA PAKAI MEAN:')
print(f'   - Karyawan merasa "dibohongi" karena gajinya di bawah "rata-rata"')
print(f'   - Benchmark industri menjadi tidak fair')
print(f'   - Executive 5 orang (gaji 80-200jt) menggelembungkan mean perusahaan')
print()
print(f'3. REKOMENDASI PER DEPARTEMEN:')
for dept in dept_order:
    row = df_summary.loc[dept]
    print(f'   - {dept:12s}: Pakai {row["rekomendasi"]:12s} (skewness={row["skewness"]:.2f})')
print()
print(f'4. DEPARTEMEN PALING FAIR (distribusi seragam): Operations')
print(f'   (gap mean-median hanya {df_summary.loc["Operations","gap_persen"]:.1f}%)')

## 📌 Takeaways

- Gaji perusahaan **selalu right-skewed** karena ada level hierarki
- Mean gaji perusahaan **diinflasi** oleh gaji eksekutif — jangan pakai untuk benchmark
- Median lebih adil untuk karyawan, trimmed mean lebih informatif untuk manajemen
- Per-departemen analysis memberikan gambaran lebih jujur dari overall mean
- Alternative: Laporkan mean PER LEVEL (junior, senior, lead) daripada mean perusahaan